# Customer Churn Prediction — Reproducible Benchmark & Retention Economics

Standalone Google Colab walkthrough evaluating Baseline Random Forest, Class-Weighted Logistic Regression, and an early-stopped MLP Neural Network on the IBM Telco Customer Churn dataset (`7,043` raw records, `7,032` usable after removing 11 blank `TotalCharges` entries).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score, brier_score_loss
)

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
raw_df = pd.read_csv(DATA_URL)
print(f"Raw shape: {raw_df.shape}")

## 1. Leak-Free Preprocessing (Split Before Scale)

`train_test_split` runs before `StandardScaler.fit_transform` so test-set mean and variance never leak into training.

In [ ]:
df = raw_df.drop(columns=["customerID"], errors="ignore").copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
dropped = int(df["TotalCharges"].isna().sum())
df = df.dropna(subset=["TotalCharges"])

X = df.drop(columns=["Churn"])
y = LabelEncoder().fit_transform(df["Churn"])

cat_cols = X.select_dtypes(include=["object", "string"]).columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
X_train, X_test = X_train.copy(), X_test.copy()

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"Dropped blank TotalCharges: {dropped} | Train: {X_train.shape} | Test: {X_test.shape} | Churn rate: {y.mean():.1%}")

## 2. 5-Fold Stratified CV & Holdout Evaluation

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
lr = LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000)
mlp_search = RandomizedSearchCV(
    MLPClassifier(max_iter=1000, early_stopping=True, random_state=42),
    {
        "hidden_layer_sizes": [(64, 32), (128, 64, 32), (32, 16)],
        "activation": ["relu", "tanh"],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate_init": [0.001, 0.01],
    },
    n_iter=5, cv=3, scoring="f1_macro", random_state=42, n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, est in [("RF Baseline", rf), ("Balanced LR", lr)]:
    scores = cross_validate(est, X_train, y_train, cv=cv, scoring=["roc_auc", "recall"])
    print(f"{name} 5-Fold CV -> ROC-AUC: {scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f} | Recall: {scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}")

rf.fit(X_train, y_train)
lr.fit(X_train, y_train)
mlp = mlp_search.fit(X_train, y_train).best_estimator_

rows = []
models = {"RF Baseline": rf, "Balanced LR": lr, "MLP NN": mlp}
for name, m in models.items():
    pred = m.predict(X_test)
    prob = m.predict_proba(X_test)[:, 1]
    rep = classification_report(y_test, pred, output_dict=True)["1"]
    fpr, tpr, _ = roc_curve(y_test, prob)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, pred), 3),
        "Precision": round(rep["precision"], 2),
        "Recall": round(rep["recall"], 3),
        "F1": round(rep["f1-score"], 2),
        "ROC-AUC": round(auc(fpr, tpr), 3),
        "PR-AUC": round(average_precision_score(y_test, prob), 3),
        "Brier": round(brier_score_loss(y_test, prob), 3),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    })
results_df = pd.DataFrame(rows)
results_df

## 3. Diagnostic Curves & Retention Economics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for name, m in models.items():
    prob = m.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[0].plot(fpr, tpr, label=f"{name} ({auc(fpr, tpr):.3f})")
    prec, rec, _ = precision_recall_curve(y_test, prob)
    axes[1].plot(rec, prec, label=f"{name} ({average_precision_score(y_test, prob):.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_title("ROC Curve"); axes[0].legend()
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", label="Baseline")
axes[1].set_title("Precision-Recall Curve"); axes[1].legend()

coefs = pd.Series(lr.coef_[0], index=X_train.columns)
top10 = coefs.reindex(coefs.abs().sort_values(ascending=False).head(10).index).sort_values()
top10.plot.barh(ax=axes[2], color=["#C0392B" if v > 0 else "#2980B9" for v in top10])
axes[2].axvline(0, color="black", lw=1)
axes[2].set_title("Top 10 LR Log-Odds")
plt.tight_layout(); plt.show()

def net_profit(tp, fp, save_rate, ltv=780.0, cost=50.0):
    return tp * save_rate * ltv - (tp + fp) * cost

rf_row = results_df.loc[results_df["Model"] == "RF Baseline"].iloc[0]
lr_row = results_df.loc[results_df["Model"] == "Balanced LR"].iloc[0]
for sr in (0.30, 0.50):
    diff = net_profit(lr_row.TP, lr_row.FP, sr) - net_profit(rf_row.TP, rf_row.FP, sr)
    print(f"Save rate {sr:.0%}: Balanced LR incremental net profit vs RF = +${diff:,.0f}")